# ReleaseBot - Week 1 API Notebook
**CoreSmart.AI · 17-Week GenAI Developer Program · Week 1**

Run ReleaseBot on your own machine, then use this notebook to exercise every endpoint. Each endpoint is shown two ways:

- **cURL** - a copy-paste reference for your own terminal (shown in markdown, not run here).
- **Python** - a `requests` cell you run right here. Works on macOS, Linux, and Windows alike, and is the path this notebook uses.

> Prefer Colab? Use `releasebot_colab.ipynb` instead, which sets the app up and runs it for you inside Colab.

---
### Before you start
1. Server running: `uvicorn app.main:app --reload`
2. `.env` filled in with `OPENAI_API_KEY`, `SMTP_SENDER`, `SMTP_PASSWORD`
3. Run the **Setup** cell below once.


In [ ]:
# Setup - run this cell first
import requests, json, textwrap

BASE      = 'http://localhost:8000'
RECIPIENT = 'training@coresmart.ai'  # change to your email

# Full demo notes (used in Python cells)
DEMO_NOTES = textwrap.dedent("""
    v2.4 - Fixed login retry loop on Safari that caused users to be locked out after 3 failed attempts.
    Adjusted session token TTL from 1h to 4h to reduce re-authentication friction.
    Added experimental dark mode toggle in user settings (opt-in only).
    Patched XSS vulnerability in the comment renderer - update strongly recommended.
    Bumped Node.js runtime from 18.x to 20.11 LTS.
    Deprecated the legacy /v1/auth endpoint; removal scheduled for v3.0 in Q3.
""").strip()

print('Setup complete.')
print('BASE      :', BASE)
print('RECIPIENT :', RECIPIENT)

---
## 1 · Health Check - `GET /health`
Confirms the server is alive and shows which model is loaded.

**cURL (any terminal):**
```bash
curl -s http://localhost:8000/health
```
*The Python cell below runs on every OS and is the path we use here.*

In [ ]:
# Health check - Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

---
## 2 · Streaming Summary - `POST /summarize-stream`

The model streams its response token-by-token as **Server-Sent Events (SSE)**.

Each frame: `data: {"delta": "..."}` &nbsp;·&nbsp; Final frame: `data: [DONE]`

> No tool call here - pure streaming text output.

**cURL (bash / zsh / Git Bash):**
```bash
curl -s -N -X POST http://localhost:8000/summarize-stream \
  -H 'Content-Type: application/json' \
  -d '{"release_notes": "v2.4 - Fixed login retry on Safari. Extended session TTL to 4h. Added dark mode."}'
```
*On Windows `cmd`, escape the inner quotes with `\"` or just use the Python cell below (works everywhere).*

In [ ]:
# Streaming - Python (reads SSE frames token by token)
payload = {'release_notes': DEMO_NOTES}

print('-- Streaming output --\n')
with requests.post(f'{BASE}/summarize-stream', json=payload, stream=True) as resp:
    resp.raise_for_status()
    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode() if isinstance(raw_line, bytes) else raw_line
        if not line.startswith('data: '):
            continue
        payload_str = line[6:]
        if payload_str == '[DONE]':
            print('\n\n-- Stream complete --')
            break
        try:
            chunk = json.loads(payload_str)
            if 'delta' in chunk:
                print(chunk['delta'], end='', flush=True)
            if 'error' in chunk:
                print(f'\n[ERROR] {chunk["error"]}')
        except json.JSONDecodeError:
            pass

---
## 3 · Structured Summary + Tool Call - `POST /summarize`

Single round-trip flow (one API call, no `role: "tool"` message, no second model turn):
1. Send release notes + tool schema, with `tool_choice` forcing `send_email` → model returns a tool call whose args are the structured output
2. We run `send_email` (the actual SMTP send) with those args
3. We build `ReleaseSummary` directly from the tool args and return both summary + tool call log

Response shape:
```json
{
  "summary":    { "headline": "...", "bullets": [...], "risk_level": "low|med|high" },
  "tool_calls": [ { "name": "send_email", "input": {...}, "result": {...} } ]
}
```

> Make sure your `.env` has valid `SMTP_SENDER` + `SMTP_PASSWORD` before running.

**cURL (bash / zsh / Git Bash):**
```bash
curl -s -X POST http://localhost:8000/summarize \
  -H 'Content-Type: application/json' \
  -d '{"release_notes": "v2.4 - Fixed login retry on Safari. Patched XSS in comment renderer. Bumped Node to 20 LTS.", "recipient": "training@coresmart.ai"}'
```
*On Windows `cmd`, escape the inner quotes with `\"` or use the Python cell below.*

In [ ]:
# Structured + tool call - Python
payload = {'release_notes': DEMO_NOTES, 'recipient': RECIPIENT}

print('Calling POST /summarize ...')
r = requests.post(f'{BASE}/summarize', json=payload)

if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()

    print('\n-- Summary --')
    s = data['summary']
    print(f"Headline   : {s['headline']}")
    print(f"Risk level : {s['risk_level']}")
    print('Bullets    :')
    for b in s['bullets']:
        print(f'  - {b}')

    print('\n-- Tool calls --')
    for tc in data['tool_calls']:
        print(f"Tool   : {tc['name']}")
        print(f"Input  : {json.dumps(tc['input'], indent=2)}")
        print(f"Result : {json.dumps(tc['result'], indent=2)}")

---
## 4 · Full Raw Response
Shows the complete JSON as returned - useful for debugging.

In [ ]:
# Full raw response dump
r = requests.post(f'{BASE}/summarize', json={'release_notes': DEMO_NOTES, 'recipient': RECIPIENT})
print(json.dumps(r.json(), indent=2))

---
## 5 · Failure Mode - Empty Release Notes
Pydantic enforces `min_length=1`. Server returns **422** before the model is called - no tokens spent.

**cURL (bash / zsh / Git Bash):**
```bash
curl -s -X POST http://localhost:8000/summarize \
  -H 'Content-Type: application/json' \
  -d '{"release_notes": "", "recipient": "you@example.com"}'
```
*On Windows `cmd`, escape the inner quotes with `\"` or use the Python cell below.*

In [ ]:
# Failure: empty notes - Python
r = requests.post(f'{BASE}/summarize', json={'release_notes': '', 'recipient': RECIPIENT})
print(f'Status: {r.status_code}  (expected 422)')
print(json.dumps(r.json(), indent=2))

---
## 6 · Failure Mode - Missing Recipient
`/summarize` requires a `recipient`. Without it the server returns **422** before any API call.

**cURL (bash / zsh / Git Bash):**
```bash
curl -s -X POST http://localhost:8000/summarize \
  -H 'Content-Type: application/json' \
  -d '{"release_notes": "v2.4 - Some changes."}'
```
*On Windows `cmd`, escape the inner quotes with `\"` or use the Python cell below.*

In [ ]:
# Failure: missing recipient - Python
r = requests.post(f'{BASE}/summarize', json={'release_notes': DEMO_NOTES})
print(f'Status: {r.status_code}  (expected 422)')
print(json.dumps(r.json(), indent=2))

---
## 7 · OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs - try the endpoints live in the browser:

In [ ]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank" style="font-size:15px">'
             'Open Swagger UI: http://localhost:8000/docs</a>'))